In [ ]:
# %% [markdown]
# # HANF-Net Lite - Feature Analysis
# ## Correlation Heatmaps & Feature Selection

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load features
df = pd.read_csv("../data/processed/final_features.csv")
print(f"Shape: {df.shape}")
print(f"Features: {list(df.columns)}")

# %%
# Select only numeric feature columns (exclude target and ID columns)
feature_cols = [col for col in df.columns if col not in [
    'No.', 'Gender', 'Age', 'Payer Zone', 'Insurance Company', 
    'Treatment Type', 'Disease Category', 'Original Status', 
    'Outcome', 'TPA Remarks', 'Approval Rate', 'Approved Amt (₹)',
    'Insurer_Disease', 'Age_Group'
]]

target = 'Coverage'

# %%
# Correlation matrix
plt.figure(figsize=(20, 16))
corr_matrix = df[feature_cols + [target]].corr()

# Plot heatmap
sns.heatmap(corr_matrix, annot=False, cmap='RdYlBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

# %%
# Correlation with target
target_corr = corr_matrix[target].sort_values(ascending=False)
print("\n📊 Top 15 Features Correlated with Coverage:")
print(target_corr.head(15))

print("\n📉 Bottom 15 Features (Negative Correlation):")
print(target_corr.tail(15))

# %%
# Find highly correlated features (redundancy)
corr_matrix_features = corr_matrix[feature_cols].loc[feature_cols]

# Get upper triangle
upper = corr_matrix_features.where(np.triu(np.ones(corr_matrix_features.shape), k=1).astype(bool))

# Find features with correlation > 0.8
high_corr = [(column, row, upper[column][row]) 
             for column in upper.columns 
             for row in upper.index 
             if abs(upper[column][row]) > 0.8]

print("\n🔴 Highly Correlated Feature Pairs (|r| > 0.8) - Consider dropping one:")
for col, row, val in high_corr:
    print(f"  {col}  ↔  {row}  :  {val:.3f}")

# %%
# Feature importance using Random Forest
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Prepare data
X = df[feature_cols].fillna(0)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train RF
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\n🔥 Feature Importance (Random Forest):")
print(importance.head(20))

# %%
# Plot feature importance
plt.figure(figsize=(10, 8))
plt.barh(importance.head(20)['feature'], importance.head(20)['importance'])
plt.xlabel('Importance')
plt.title('Top 20 Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# %%
# FINAL FEATURE SELECTION
# Based on correlation and importance, we'll select the best features

selected_features = [
    # Keep these based on importance and low redundancy
    'ID_Coverage',           # High importance
    'Bill_Z_Disease',        # Financial
    'Disease_Avg_Cov',       # Disease pattern
    'Insurer_Generosity',    # Insurer pattern
    'Cost_per_Day',          # Financial ratio
    'Combined_Risk',         # Fuzzy risk
    'LOS_Z_Disease',         # LOS pattern
    'Bill_Pct_Disease',      # Percentile
    'Zone_Bias',             # Geographic
    'ID_Deviation',          # Interaction
    'Disease_Risk',          # Risk
    'Log_Bill',              # Transformed
    'Age_Group_Encoded',     # Demographic
    'Insurer_Rejection_Rate', # Insurer behavior
    'Gender_Encoded'         # Demographic
]

print(f"\n✅ Selected {len(selected_features)} features for model:")
for i, feat in enumerate(selected_features, 1):
    print(f"  {i:2d}. {feat}")

# %%
# Save selected features
df_selected = df[selected_features + [target]]
df_selected.to_csv("../data/processed/model_ready_features.csv", index=False)
print("\n💾 Saved model-ready features to: data/processed/model_ready_features.csv")